# 01 — Sanity

Two questions, in order. The second is meaningless if the first fails.

| | |
|---|---|
| **Part 1** | Is the cluster up? Every component present, healthy, scraped |
| **Part 2** | Is each component *behaving*? Confirmed by its own metrics under light load |

Light load on purpose. This is not a benchmark — it is the check you run
before trusting one. `02-capacity` is where load matters.

**Needs** `make ssh` (tunnel) and `make observability` (Prometheus, Grafana).

Numbers come from **Prometheus**, not from scraping pods: a percentile over
histogram buckets is a percentile; ten samples sorted in Python is not.


In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os
sys.path.insert(0, '..')

from utils import Cluster, Report, filler, percentile
from prom import Prom, Grafana, COMPONENT_CHECKS, LATENCY_HISTOGRAMS, panel_id
from slos import table_rows

cluster = Cluster()
prom    = Prom()                       # :31490
grafana = Grafana(password=os.environ.get('GRAFANA_PASSWORD', ''))
report  = Report('Dodona 01 — sanity', 'is it up, and is it behaving?')

report.objective(
    'Is every component up, and does each one behave as designed under a '
    'light load — as judged by its own metrics?',
    falsified_by='any component missing, or a metric that contradicts its design')


---
# Part 1 — Is the cluster up?

Presence and health. Nothing here measures behaviour yet.


## 1.1 Do the endpoints answer?


In [ ]:
report.heading('Part 1 — is the cluster up?')
up = cluster.endpoints()
up['prometheus'] = prom.up()
for name, ok in up.items():
    report.verdict(ok, name)

if all(up.values()):
    report.conclusion('Every endpoint answers, so the metrics below are real.')
else:
    down = [n for n, ok in up.items() if not ok]
    report.conclusion(f'DOWN: {", ".join(down)}. `make ssh` for the tunnel, '
                      '`make observability` for Prometheus and Grafana.')


## 1.2 Is Prometheus actually scraping everything?

An empty Grafana panel and a missing scrape target look identical. This
separates them: zero series for a prefix means **nothing exports it**, which
is a different problem from a metric that is genuinely zero.


In [ ]:
prefixes = {
    'vllm:': 'the engines',
    'orch_': 'our gateway — guardrails, admission, placement, hop',
    'DCGM_FI': 'GPU telemetry',
    'kube_': 'kube-state-metrics — replicas, for KEDA',
    'keda_': 'the autoscaler',
}
rows, missing = [], []
for prefix, what in prefixes.items():
    n = prom.series_exist(prefix)
    rows.append([prefix, n, what])
    if n == 0:
        missing.append(prefix)
report.table(['metric prefix', 'series', 'who emits it'], rows,
             note='0 means nothing scrapes it — not that the value is zero.')
report.verdict(not missing, f'all {len(prefixes)} metric families are being scraped')
report.conclusion(
    'Every family is present, so an empty panel later means a real zero.'
    if not missing else
    f'Missing: {", ".join(missing)}. Any panel built on those is empty for a '
    'reason that has nothing to do with the cluster behaving.')


## 1.3 Capacity — what one slice actually got

The denominator of every concurrency claim. The **card** has ~64 GiB of KV;
a **12 GiB slice** has ~4.1 after weights and CUDA graphs. Quoting the card's
figure for a slice overstates concurrency by about 15x.

From `make cluster`'s capacity checks, or:
`make logs | grep -E 'Available KV cache|Maximum concurrency'`


In [ ]:
KV_GIB_PER_POD, SEQ_PER_POD, DECODE_REPLICAS = 4.10, 4.7, 1
SLICE_GIB, UTIL, WEIGHTS_GIB = 12.0, 0.90, 5.2
overhead = SLICE_GIB * UTIL - WEIGHTS_GIB - KV_GIB_PER_POD
pool = SEQ_PER_POD * DECODE_REPLICAS

report.heading('1.3 Capacity')
report.table(['quantity', 'value', 'source'], [
    ['slice x util', f'{SLICE_GIB*UTIL:.2f} GiB', 'HAMi gpumem x gpu-memory-utilization'],
    ['weights', f'{WEIGHTS_GIB:.2f} GiB', 'Qwen2.5-7B AWQ int4'],
    ['activations + CUDA graphs', f'{overhead:.2f} GiB', 'budget - weights - KV'],
    ['KV cache', f'{KV_GIB_PER_POD:.2f} GiB', 'pod log'],
    ['seqs per pod @16k', f'{SEQ_PER_POD:.1f}', 'pod log'],
    ['decode pool', f'{pool:.1f}', 'seqs x replicas'],
])
report.bar('Where the 12 GiB slice goes', ['weights', 'activations + graphs', 'KV'],
           {'GiB': [WEIGHTS_GIB, max(overhead, 0), KV_GIB_PER_POD]}, ylabel='GiB',
           note='only the third bar holds sequences')
report.verdict(pool < 20, f'a pool of {pool:.1f} is small enough for KV to bind')
report.conclusion(
    f'KV gets {KV_GIB_PER_POD:.2f} of {SLICE_GIB*UTIL:.2f} usable GiB — {pool:.1f} '
    'concurrent full-length sequences. That scarcity is what makes an admission '
    'policy able to decide anything at all.')


---
# Part 2 — Is each component behaving?

A light load, then every component judged by its own metrics.

Six components: **prefill/decode roles · scheduler · KV hop · admission ·
routing · KEDA**, plus the latency histograms.

Light deliberately — enough traffic to make the counters move, not enough to
saturate. Saturation is `02-capacity`'s job.


## 2.0 Generate the light load

A mix: short requests, and long unique ones so the KV hop has something
worth transferring. Unique matters — a repeated prompt is served from
decode's own cache and nothing crosses between pods.


In [ ]:
report.heading('Part 2 — is each component behaving?')
import random
random.seed(1)

sent, failed = 0, {}
for i in range(8):
    r = cluster.chat('Reply with the single word: ok', max_tokens=8)
    sent += 1
    if '_error' in r:
        failed[r['_error']] = failed.get(r['_error'], 0) + 1
for i in range(4):
    tag = f'{i}-{random.random()}'
    r = cluster.chat(f'[{tag}] ' + filler(4000) + '\n\nReply with: ok',
                     max_tokens=8, timeout=180)
    sent += 1
    if '_error' in r:
        failed[r['_error']] = failed.get(r['_error'], 0) + 1

report.table(['', 'count'], [['requests sent', sent],
                            *[[f'failed: {k}', v] for k, v in failed.items()]],
             note='8 short + 4 long unique. Failures here are gateway decisions.')
import time; time.sleep(20)   # let Prometheus scrape (15s interval)
report.conclusion(f'{sent} requests sent, {sum(failed.values())} refused. '
                  'Waited one scrape interval so the counters below are current.')


## 2.1 Every component, by its own metrics

Read from Prometheus. Each row carries what the number *means* — a metric
name alone tells you nothing about whether its value is good.


In [ ]:
for group, checks in COMPONENT_CHECKS.items():
    rows = []
    for label, expr, meaning in checks:
        got = prom.query(expr)
        value = f'{got[0][1]:,.4g}' if got else 'no data'
        rows.append([label, value, meaning])
    report.table([group, 'value', 'what it means'], rows)


## 2.2 Latency, as percentiles over histograms

`histogram_quantile` over the bucket series. Ten samples sorted in Python
would give a p99 that is just the maximum of ten.

| | |
|---|---|
| **TTFT** | time to first token — a voice agent's dead air |
| **ITL** | inter-token latency (`time_per_output_token`) — how it reads aloud |
| **E2E** | the whole request |
| **queue** | time waiting before the GPU touched it |


In [ ]:
report.heading('2.2 Latency')
rows, series = [], {}
for name in ('ttft', 'tpot', 'e2e', 'queue', 'prefill', 'decode'):
    q = prom.quantiles(name, qs=(0.5, 0.9, 0.99))
    if q:
        rows.append([name.upper() if name in ('ttft','e2e') else name,
                     f'{q.get(0.5, 0):.3f}', f'{q.get(0.9, 0):.3f}', f'{q.get(0.99, 0):.3f}'])
        series[name] = [q.get(0.5, 0), q.get(0.9, 0), q.get(0.99, 0)]

if rows:
    report.table(['metric', 'p50 s', 'p90 s', 'p99 s'], rows,
                 note='From histogram_quantile over the bucket series, 5m window.')
    report.bar('Latency percentiles', ['p50', 'p90', 'p99'], series,
               ylabel='seconds', log=True,
               note='ITL is tpot — per output token, so orders of magnitude below E2E')
    ttft = prom.quantiles('ttft', qs=(0.95,))
    if ttft:
        report.slo('ttft_p95', ttft[0.95])
else:
    report.verdict(False, 'no latency histograms — did any request complete?')


## 2.3 KV usage over time

One scrape cannot tell a steady state from a spike you caught. This is the
series, which is the whole reason for going through Prometheus.


In [ ]:
report.heading('2.3 KV over time')
ranged = prom.query_range('vllm:kv_cache_usage_perc', minutes=15)
if ranged:
    import datetime as _dt
    xs = [p[0] for p in ranged[0][1]]
    series = {}
    for labels, points in ranged:
        pod = labels.get('pod', labels.get('instance', '?'))[:22]
        series[pod] = [v for _, v in points]
    n = min(len(v) for v in series.values())
    report.line('KV cache usage over the last 15 minutes',
                list(range(n)), {k: v[:n] for k, v in series.items()},
                xlabel='scrape (15s apart)', ylabel='kv_cache_usage_perc',
                hlines={'admission threshold 0.90': 0.90},
                note='below the line the kv_threshold policy is inert')
    peak = max(max(v) for v in series.values())
    report.conclusion(
        f'Peak KV over 15 minutes was {peak:.1%}. '
        + ('It crossed the 0.90 threshold, so admission had something to decide.'
           if peak >= 0.90 else
           'Never near 0.90, so a kv_threshold policy would have been inert on this '
           'load — which is expected here and is what 02-capacity exists to fix.'))
else:
    report.verdict(False, 'no KV series in Prometheus')


## 2.4 The panels themselves

Rendered by Grafana, not redrawn. A screenshot of the chart the team already
watches needs no explanation of how it was derived.

Needs `GRAFANA_PASSWORD` (printed by `make observability`) and the renderer
deployment. Falls back to links.


In [ ]:
from pathlib import Path
shots = Path('screenshots'); shots.mkdir(exist_ok=True)
wanted = [('dodona-vllm', 'vllm', 'GPU KV cache'),
          ('dodona-vllm', 'vllm', 'TTFT'),
          ('dodona-mooncake', 'mooncake', 'DELIVERED'),
          ('dodona-gateway', 'gateway', 'shed')]
saved = []
for uid, dash, title in wanted:
    pid = panel_id(dash, title)
    if pid is None:
        continue
    out = grafana.render(uid, pid, shots / f'{dash}-{title.lower()}.png', minutes=30)
    if out:
        saved.append(out.name)

if saved:
    report.verdict(True, f'rendered {len(saved)} Grafana panels into screenshots/')
    report.conclusion('Panels saved: ' + ', '.join(saved))
    from IPython.display import Image, display
    for s in saved:
        display(Image(filename=str(shots / s)))
else:
    report.text('No panels rendered. Set GRAFANA_PASSWORD and check renderer.yaml '
                'is deployed; /render returns 302 without credentials.')
    from prom import GRAFANA_URL
    report.links([(f'{d} dashboard', f'{GRAFANA_URL}/d/{u}?from=now-30m&to=now')
                  for u, d, _ in dict.fromkeys((u, d, '') for u, d, _ in wanted)])


## Save


In [ ]:
path  = report.save('01-sanity')
fails = report.failures()
print('wrote', path)
print()
if fails:
    print('NOT READY — these are open:')
    for f in fails:
        print('  -', f)
else:
    print('Cluster is up and every component is behaving. Next: 02-capacity')
